In [ ]:
# ---------------------------------------------------------------------------
# Dice Similarity Coefficient (DSC) and True Positive Rate (TPR) grouped bar plot with 
# mean and ± standard deviation (SD) band
# ---------------------------------------------------------------------------
# This script produces a publication-ready grouped bar plot comparing two
# segmentation performance metrics — Dice Similarity Coefficient (DSC) and
# True Positive Rate (TPR) — across multiple scan sessions. Each session is
# represented by a pair of bars, one per metric. A horizontal dashed line
# marks the mean DSC across all sessions, and a shaded band spans ± one
# standard deviation around that mean, providing a visual summary of
# cross-session variability. The plot is saved as a vector PDF suitable for
# direct inclusion in a manuscript.
# ---------------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D   # Used to create custom line entries in the legend
from matplotlib.patches import Patch  # Used to create custom patch (rectangle) entries in the legend

# ---------------------------------------------------------------------------
# Dataset labels shown on the x-axis.
datasets = ["Sav_S1", "Sav_S2", "Mt_S1", "Mt_S2", "Col_S1", "Col_S2"]

# DSC measures the spatial overlap between predicted and ground-truth segmentation masks.
# Range: [0, 1], where 1 = perfect overlap.
# Replace with your own DSC values; must be the same length as `datasets`.
DSC = [0.78, 0.90, 0.75, 0.57, 0.75, 0.76]

# TPR / Sensitivity / Recall per dataset.
# TPR = TP / (TP + FN) — proportion of actual positives correctly identified.
# Range: [0, 1], where 1 = no missed detections.
# Replace with your own TPR values; must be the same length as `datasets`.
TPR = [0.91, 0.93, 0.88, 0.41, 0.87, 0.91]

# ---------------------------------------------------------------------------
# Descriptive statistics for DSC
# ---------------------------------------------------------------------------
# Compute the arithmetic mean of DSC values across all datasets.
mean_dsc = float(np.mean(DSC))

# Compute the sample SD (ddof=1 applied Bessel's correction,
# dividing by N-1 instead of N — appropriate when treating these as a sample
# drawn from a larger population).
sd_dsc = float(np.std(DSC, ddof=1))

# ---------------------------------------------------------------------------
# Figure & axes setup
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.8))  # Width=8in, Height=4.8in

# Numeric positions for each group of bars on the x-axis (0, 1, 2, …, 5)
x = np.arange(len(datasets))

# Width of each individual bar. Two bars share one group slot, so each bar
# occupies roughly half the inter-group spacing.
bar_w = 0.38

# ---------------------------------------------------------------------------
# Draw grouped bars
# ---------------------------------------------------------------------------
# DSC bars: shifted LEFT of each group centre by half the bar width.
bars_dsc = ax.bar(x - bar_w/2, DSC, width=bar_w, label="DSC", color="cornflowerblue")

# TPR bars: shifted RIGHT of each group centre by half the bar width.
bars_tpr = ax.bar(x + bar_w/2, TPR, width=bar_w, label="TPR", color="gold")

# ---------------------------------------------------------------------------
# Axes formatting
# ---------------------------------------------------------------------------
# Place one tick per dataset group and rotate labels to avoid overlap.
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha="right")

# Set y-axis range slightly above 1.0 to leave headroom for the legend and
# the SD band annotation without clipping any bar or overlay element.
ax.set_ylim(0, 1.3)
ax.set_ylabel("Score")

# ---------------------------------------------------------------------------
# Mean DSC reference line and ± SD shaded band
# ---------------------------------------------------------------------------
# Clamp band boundaries to [0, 1.05] so the shading never extends below zero
# or unrealistically above the maximum possible score.
band_low = max(0,    mean_dsc - sd_dsc)   # lower edge of the ± SD band
band_hi  = min(1.05, mean_dsc + sd_dsc)   # upper edge of the ± SD band

# axhspan draws a full-width horizontal band between band_low and band_hi.
# that is visible without obscuring the bars beneath it.
band = ax.axhspan(band_low, band_hi, facecolor="black", alpha=0.15)

# axhline draws a single horizontal dashed line at the mean DSC value.
line = ax.axhline(mean_dsc, linestyle="--", color="black")

# ---------------------------------------------------------------------------
# Custom legend
# ---------------------------------------------------------------------------
# Because the SD band and mean line are not standard bar objects, we build
# the legend manually using proxy artists (Patch and Line2D).
handles = [
    bars_dsc,                                        # BarContainer → auto-detected
    bars_tpr,                                        # BarContainer → auto-detected
    Patch(facecolor="black", alpha=0.15),            # Proxy for the SD band
    Line2D([0], [0], linestyle="--", color="black"), # Proxy for the mean line
]
labels = [
    "DSC",
    "TPR",
    f"± SD ({sd_dsc:.2f})",          # Shows the actual SD value for quick reference
    f"Mean DSC = {mean_dsc:.2f}",    # Shows the actual mean value for quick reference
]

# Place the legend in the upper-right corner; frameon=False removes the box border
ax.legend(handles, labels, loc="upper right", frameon=False)

# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------
plt.tight_layout()  # Automatically adjust subplot margins to prevent label clipping

# Save as vector PDF at 300 DPI; bbox_inches="tight" ensures nothing is cropped.
plt.savefig("dsc_tpr_barplot.pdf", dpi=300, bbox_inches="tight")

plt.show() 